# Solution 8: Comparing ERA5 Reanalysis with Climate DT Projections

In this advanced exercise, you'll:
1. Retrieve a historical ERA5 reanalysis field from CDS
2. Retrieve a Climate DT projection field
3. Regrid both to a common grid
4. Plot them side-by-side and compute the difference

This demonstrates how to compare present-day observations with future projections.

> **Prerequisites**: These notebooks require the `cdsapi` package which is not included in the project requirements. Install it with:
> 
> ```bash
> uv pip install cdsapi
> ```
> 
> You also need a CDS API key configured in `~/.cdsapirc`. Register at [https://cds.climate.copernicus.eu](https://cds.climate.copernicus.eu) to obtain one.

## Authentication

In [ ]:
%%capture cap
%run ../../desp-authentication.py

In [ ]:
output_1 = cap.stdout.split('}\n')
access_token = output_1[-1][0:-1]

## Imports

In [ ]:
import earthkit.data
import earthkit.plots
import earthkit.regrid
import numpy as np
import matplotlib.pyplot as plt
import os

## Live Request Toggle

In [ ]:
LIVE_REQUEST = os.getenv("LIVE_REQUEST", "true").lower() == "true"
LIVE_REQUEST

## Step 1: Retrieve ERA5 Reanalysis Data

We retrieve 2m temperature from ERA5 for a historical date using the CDS (Climate Data Store) source.

We request directly at 1°×1° resolution so it matches our target comparison grid (no client-side regridding needed for ERA5).

**Note**: You need a CDS API key configured (`~/.cdsapirc`) to access ERA5 data.

**EXERCISE**: Change the date/time to match a specific historical event you're interested in.

In [ ]:
request_era5 = {
    "product_type": "reanalysis",
    "variable": "2m_temperature",
    "year": "2020",
    "month": "01",
    "day": "02",
    "time": "12:00",
    "area": [75, 0, 25, 50],  # N, W, S, E (Central/Eastern Europe)
    "grid": [1, 1],
}

era5_file = "../../climate-dt/data/era5-training.grib"

if LIVE_REQUEST:
    data_era5 = earthkit.data.from_source(
        "cds", 
        "reanalysis-era5-single-levels",
        request_era5,
    )
    data_era5.to_target("file", era5_file)
else:
    data_era5 = earthkit.data.from_source("file", era5_file)

data_era5.ls()

## Step 2: Retrieve Climate DT Projection Data

Now we retrieve the same variable (2m temperature) from the Climate DT for a future date.

In [ ]:
request_climate = {
    "activity": "projections",
    "class": "d1",
    "dataset": "climate-dt",
    "date": "20400102",
    "time": "1200",
    "experiment": "SSP3-7.0",
    "expver": "0001",
    "generation": "2",
    "levtype": "sfc",
    "model": "IFS-NEMO",
    "param": "167",
    "realization": "1",
    "resolution": "standard",
    "stream": "clte",
    "type": "fc",
}

climate_file = "../../climate-dt/data/climate-dt-comparison-training.grib"

if LIVE_REQUEST:
    data_climate = earthkit.data.from_source(
        "polytope", "destination-earth", request_climate,
        address="polytope.mn5.apps.dte.destination-earth.eu",
        stream=False,
    )
    data_climate.to_target("file", climate_file)
else:
    data_climate = earthkit.data.from_source("file", climate_file)

data_climate.ls()

## Step 3: Regrid to a Common Grid

To compute differences, both datasets must be on the same grid. We regrid both to a common 1 x 1 degree regular lat-lon grid.

In [ ]:
common_grid = {"grid": [1, 1]}

# ERA5 was already requested at 1x1, no regridding needed
era5_regridded = data_era5

# Regrid Climate DT (from HEALPix to 1 degree regular)
climate_regridded = earthkit.regrid.interpolate(
    data_climate, out_grid=common_grid, method="linear"
)

print(f"ERA5 fields: {len(era5_regridded)}")
print(f"Climate DT regridded fields: {len(climate_regridded)}")

## Step 4: Convert to xarray for Analysis

In [ ]:
ds_era5 = era5_regridded.to_xarray()
ds_climate_global = climate_regridded.to_xarray()

# Subset Climate DT to match the ERA5 European domain [75N, 0E, 25N, 50E]
ds_climate = ds_climate_global.sel(
    latitude=slice(75, 25),
    longitude=slice(0, 50)
)

print("ERA5 shape:", ds_era5["2t"].shape)
print("Climate DT (Europe) shape:", ds_climate["2t"].shape)

## Step 5: Plot Side-by-Side Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Get temperature arrays in Celsius
era5_temp = ds_era5["2t"].squeeze().values - 273.15
climate_temp = ds_climate["2t"].squeeze().values - 273.15

# Calculate the difference (Climate DT - ERA5)
difference = climate_temp - era5_temp

# Plot ERA5
ax1 = axes[0]
im1 = ax1.imshow(era5_temp, cmap="RdYlBu_r", vmin=-30, vmax=35, 
                  extent=[0, 50, 25, 75], aspect="auto", origin="upper")
ax1.set_title("ERA5 Reanalysis (Jan 2020)")
plt.colorbar(im1, ax=ax1, label="\u00b0C")

# Plot Climate DT
ax2 = axes[1]
im2 = ax2.imshow(climate_temp, cmap="RdYlBu_r", vmin=-30, vmax=35,
                  extent=[0, 50, 25, 75], aspect="auto", origin="upper")
ax2.set_title("Climate DT SSP3-7.0 (Jan 2040)")
plt.colorbar(im2, ax=ax2, label="\u00b0C")

# Plot the difference
ax3 = axes[2]
im3 = ax3.imshow(difference, cmap="RdBu_r", vmin=-10, vmax=10,
                  extent=[0, 50, 25, 75], aspect="auto", origin="upper")
ax3.set_title("Projected Change (\u00b0C)")
plt.colorbar(im3, ax=ax3, label="\u00b0C")

plt.tight_layout()
plt.show()

## Step 6: Statistical Summary

In [ ]:
print("=== Projected Temperature Change (2040 vs 2020) ===")
print(f"Mean change: {np.nanmean(difference):.2f} \u00b0C")
print(f"Max warming: {np.nanmax(difference):.2f} \u00b0C")
print(f"Min change:  {np.nanmin(difference):.2f} \u00b0C")
print(f"Std dev:     {np.nanstd(difference):.2f} \u00b0C")

## Bonus: Regional Analysis

Subsetting to the Mediterranean region and computing average warming.

In [ ]:
# Mediterranean region (lat: 30-45, lon: 5 to 35)
lat_min, lat_max = 30, 45
lon_min, lon_max = 5, 35

# Subset using xarray's sel method
ds_era5_med = ds_era5["2t"].sel(
    latitude=slice(lat_max, lat_min),
    longitude=slice(lon_min, lon_max)
)
ds_climate_med = ds_climate["2t"].sel(
    latitude=slice(lat_max, lat_min),
    longitude=slice(lon_min, lon_max)
)

# Compute regional difference
era5_med_temp = ds_era5_med.squeeze().values - 273.15
climate_med_temp = ds_climate_med.squeeze().values - 273.15
regional_difference = climate_med_temp - era5_med_temp

regional_mean = np.nanmean(regional_difference)
print(f"Mediterranean regional mean warming: {regional_mean:.2f} °C")

## Summary

In this exercise you've learned to:
- Retrieve data from multiple sources (CDS for ERA5, Polytope for Climate DT)
- Regrid heterogeneous datasets to a common grid
- Compute and visualize differences between reanalysis and projections
- Perform basic climate change signal analysis

Key takeaways:
- Always regrid to a common grid before comparing datasets
- The Climate DT provides km-scale climate projections that can be directly compared with ERA5
- SSP3-7.0 shows significant warming signals even for the 2040 timeframe